# Lesson 1.2: Tokens, cost, temperature and reasoning effort, by hand
*Build an AI Agent from Scratch with LangGraph · Chapter 1*

**Goal:** find out where the 102 completion tokens of lesson 1.1 went, price a call by hand, and see what the two
knobs, temperature and reasoning effort, really change.

**What you will be able to do**
- count tokens with `tiktoken` and explain why the API counts more (wrappers, hidden reasoning)
- price a call from `usage`: input and output tokens at their own price per million
- explain why temperature 0 is steadier but not identical, and test a prompt more than once
- choose reasoning effort low / medium / high and see what it costs

## How to use this notebook
- Run the cells **from top to bottom** (Shift + Enter).
- Cells that start with `# calls the API` send real requests to Groq (four cells, small prompts, about 15 requests
  in total). Without a key they print `skipped` and the notebook uses the numbers from the video.
- Sampling is random, so your temperature and effort results may differ a little from the video.
- Each exercise has a check cell that prints `✓ Exercise N passed`; no check cell calls the API. Solutions at the end.
- **Colab:** `pip install tiktoken openai python-dotenv` first, and set the key with
  `from getpass import getpass; os.environ["GROQ_API_KEY"] = getpass("Groq key: ")`.

In [1]:
# Setup: run this cell first.
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          # finds a .env in this folder or any folder above it
OFFLINE = os.environ.get("NOTEBOOK_OFFLINE") == "1"   # set NOTEBOOK_OFFLINE=1 to skip every API cell
HAVE_KEY = bool(os.environ.get("GROQ_API_KEY")) and not OFFLINE
MODEL = "openai/gpt-oss-20b"
if HAVE_KEY:
    print("GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.")
else:
    print("No GROQ_API_KEY: cells marked 'calls the API' print 'skipped' and the video's numbers are used.")
    print("Get the free key in lesson 0.2, put the line GROQ_API_KEY=... in a .env file, restart the kernel.")

GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.


In [2]:
# Make the course code importable: find the repo's nightwatch/ folder above this notebook.
import sys
from pathlib import Path

here = Path.cwd()
NIGHTWATCH = next((p / "nightwatch" for p in [here, *here.parents]
                   if (p / "nightwatch" / "agent_v0.py").exists()), None)
if NIGHTWATCH:
    sys.path.insert(0, str(NIGHTWATCH))
    print("nightwatch/ found at", Path(os.path.relpath(NIGHTWATCH, here)).as_posix())
else:
    print("nightwatch/ not found: open this notebook from lessons/<lesson>/notebook/ inside the course repo.")

nightwatch/ found at ../../../nightwatch


In [3]:
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", "none"),
                base_url="https://api.groq.com/openai/v1", max_retries=8)
SYSTEM = "You are Nightwatch, the on-call assistant for the shop Brightcart. Answer in one sentence."
ALERT = "Alert: checkout error rate is 23%. What does that mean?"
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ALERT}]

## 1. Recall (lesson 1.1)
A list of messages goes in, one `assistant` message comes out. Its usage was 108 prompt + 102 completion tokens.

In [4]:
video_usage = {"prompt_tokens": 108, "completion_tokens": 102}
print("total:", video_usage["prompt_tokens"] + video_usage["completion_tokens"])

total: 210


## 2. Key words

| Word | Meaning |
|---|---|
| token | a small piece of text: a whole word or part of one |
| tokenizer | the program that cuts text into tokens and gives each one a number |
| prompt tokens | everything the model reads in a call |
| completion tokens | everything the model writes back |
| reasoning tokens | the model's hidden thinking before it answers; they count as completion tokens |
| temperature | how much chance goes into picking each next token |
| reasoning effort | how much the model may think first: `low`, `medium` or `high` |

## 3. The idea and hand traces
**Text is cut into tokens.** Common words are one token, rare words split.

| Text | Tokens | Pieces |
|---|---|---|
| `checkout` | 1 | `checkout` |
| `Brightcart` | 2 | `Bright` · `cart` |
| `Nightwatch` | 2 | `Night` · `watch` |
| `error rate is 23%` | 6 | `error` · ` rate` · ` is` · ` ` · `23` · `%` |

**Prompt side.** Our two texts: system 20 + user 14 = **34** tokens. The API counted **108**.
108 − 34 = **74** tokens we never typed: role names, markers around each message, default instructions in front.

**Completion side** (one run in the video): 102 completion tokens = **72** reasoning + **21** visible answer + **9** more
we see in neither part.

**Cost by hand** (Groq prices for gpt-oss-20b at recording time: $0.075 per million input, $0.30 per million output,
so one output token costs 4× an input token):

| Part | Tokens | Price / 1M | Cost |
|---|---|---|---|
| input | 108 | $0.075 | $0.0000081 |
| output | 102 | $0.30 | $0.0000306 |
| **total** | | | **$0.0000387** |

Output is about four fifths of the bill. A million such calls: **$38.70**.

## 4. Write it yourself
### 4.1 Count tokens (no API)
`o200k_harmony` is the tokenizer of the gpt-oss models. Predict the counts first, using the table above.

In [5]:
import tiktoken

enc = tiktoken.get_encoding("o200k_harmony")
for text in ["checkout", "Brightcart", "error rate is 23%", "Nightwatch"]:
    ids = enc.encode(text)
    print(f"{text!r:22} -> {len(ids)} tokens: {[enc.decode([i]) for i in ids]}")
n_sys, n_user = len(enc.encode(SYSTEM)), len(enc.encode(ALERT))
print("system text:", n_sys, "| user text:", n_user, "| sum:", n_sys + n_user)

'checkout'             -> 1 tokens: ['checkout']
'Brightcart'           -> 2 tokens: ['Bright', 'cart']
'error rate is 23%'    -> 6 tokens: ['error', ' rate', ' is', ' ', '23', '%']
'Nightwatch'           -> 2 tokens: ['Night', 'watch']
system text: 20 | user text: 14 | sum: 34


### 4.2 Price a call (no API)
Two prices and one line of maths on the usage numbers.

In [6]:
PRICE_IN, PRICE_OUT = 0.075, 0.30          # dollars per million tokens (Groq, gpt-oss-20b, at recording time)

def call_cost(prompt_tokens, completion_tokens):
    return prompt_tokens / 1e6 * PRICE_IN + completion_tokens / 1e6 * PRICE_OUT

cost = call_cost(108, 102)
print(f"108 x $0.075/M + 102 x $0.30/M = ${cost:.8f}")
print(f"output share of the bill: {102 * PRICE_OUT / 1e6 / cost:.0%}")
print(f"one million such calls: ${cost * 1e6:,.2f}")

108 x $0.075/M + 102 x $0.30/M = $0.00003870
output share of the bill: 79%
one million such calls: $38.70


### 4.3 What the API counts
One real call. We read prompt and completion tokens, the reasoning tokens, and the start of the hidden reasoning.

In [7]:
# calls the API (1 request)
if HAVE_KEY:
    r = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0)
    u = r.usage
    prompt_t, completion_t = u.prompt_tokens, u.completion_tokens
    details = getattr(u, "completion_tokens_details", None)
    reasoning_t = getattr(details, "reasoning_tokens", None)
    visible_t = len(enc.encode(r.choices[0].message.content))
    think = getattr(r.choices[0].message, "reasoning", None) or ""
else:
    print("skipped (no key): video numbers")
    prompt_t, completion_t, reasoning_t, visible_t = 108, 102, 72, 21
    think = 'We need to answer in one sentence. The user says: "Alert: checkout error rate is 23%. What does that mean?"'
print("prompt_tokens:", prompt_t, "| our two texts:", n_sys + n_user, "| extra:", prompt_t - (n_sys + n_user))
print("completion_tokens:", completion_t, "| reasoning:", reasoning_t, "| visible answer:", visible_t,
      "| unseen rest:", completion_t - (reasoning_t or 0) - visible_t)
print("hidden reasoning (first 120 chars):", think[:120].replace("\n", " "))
print(f"this call cost: ${call_cost(prompt_t, completion_t):.8f}")

prompt_tokens: 108 | our two texts: 34 | extra: 74
completion_tokens: 102 | reasoning: 72 | visible answer: 21 | unseen rest: 9
hidden reasoning (first 120 chars): We need to answer in one sentence. The user says: "Alert: checkout error rate is 23%. What does that mean?" We need to e
this call cost: $0.00003870


Video: 108 prompt (34 ours + 74 extra), 102 completion (72 reasoning + 21 visible + 9 unseen), $0.00003870.
Most of what the call wrote was thinking that we never read.

### 4.4 Temperature: same question, five times each
Short answer (at most five words), five times at temperature 0 and five times at temperature 1.

In [8]:
# calls the API (10 requests, tiny prompt)
from collections import Counter

q = [{"role": "user", "content": "Name one likely cause of a sudden checkout error spike. Answer in at most five words."}]
for t in (0, 1):
    if not HAVE_KEY:
        print(f"temperature {t}: skipped (no key)")
        continue
    outs = [client.chat.completions.create(model=MODEL, messages=q, temperature=t).choices[0].message.content.strip()
            for _ in range(5)]
    c = Counter(outs)
    print(f"temperature {t}: {len(c)} different answers out of 5")
    for text, k in c.most_common():
        print(f"   {k}x {text!r}")

temperature 0: 2 different answers out of 5
   4x 'Payment gateway outage.'
   1x ''


temperature 1: 4 different answers out of 5
   2x 'Payment gateway outage.'
   1x 'Payment gateway timeout.'
   1x 'Database connection timeout.'
   1x 'Database connection pool exhaustion.'


Video: temperature 0 gave `Payment gateway outage.` five times; temperature 1 gave 3 different answers out of 5.
An empty reply (`''`) can show up too: the model spent its turn on hidden reasoning and wrote no text. The live
coding run in the video hit that as well, so even temperature 0 gave two "answers" there.

**The hook: temperature 0 is steadier, not identical.** The one-sentence question from 4.3, eight times at temperature 0
(video run; we do not repeat these 8 calls here):

| Wording | Runs | Completion tokens |
|---|---|---|
| "It means that nearly one in four checkout attempts are failing, indicating ..." | 5 | 102 |
| "It means that nearly one in four checkout attempts are failing—23 % of all ..." | 3 | 113 |

Both start the same and split right after *failing*: one pick changed, and everything after it changed too.
A longer answer has more room to drift, so test an agent prompt several times, not once.

In [9]:
hook_runs = [102, 102, 113, 102, 102, 113, 113, 102]      # completion tokens of the 8 runs in the video
print(len(set(hook_runs)), "different answers out of", len(hook_runs), "|", Counter(hook_runs))

2 different answers out of 8 | Counter({102: 5, 113: 3})


### 4.5 Reasoning effort
By hand first: 1,400 requests per second ÷ 250 per replica = 5.6, and you cannot run 0.6 of a replica, so **6**.

In [10]:
import math
print(1400 / 250, "->", math.ceil(1400 / 250), "replicas")

5.6 -> 6 replicas


In [11]:
# calls the API (3 requests)
hard = [{"role": "user", "content": "A service has 3 replicas, each handles 250 requests per second. Traffic is 1,400 "
                                    "requests per second. How many replicas are needed? Answer with the number only."}]
effort_tokens = {"low": 39, "medium": 82, "high": 334}      # video numbers, replaced by yours below
for effort in ("low", "medium", "high"):
    if not HAVE_KEY:
        print(f"{effort:6} -> skipped (no key), video: answer '6', reasoning_tokens {effort_tokens[effort]}")
        continue
    r = client.chat.completions.create(model=MODEL, messages=hard, temperature=0, reasoning_effort=effort)
    d = r.usage.completion_tokens_details
    effort_tokens[effort] = getattr(d, "reasoning_tokens", None)
    print(f"{effort:6} -> answer {r.choices[0].message.content.strip()!r:6} completion_tokens {r.usage.completion_tokens:4}"
          f"  reasoning_tokens {effort_tokens[effort]}")

low    -> answer '6'    completion_tokens   49  reasoning_tokens 39


medium -> answer '6'    completion_tokens   92  reasoning_tokens 82


high   -> answer '6'    completion_tokens  344  reasoning_tokens 334


Video: all three answered `6`, with **39**, **82** and **334** reasoning tokens. High used almost nine times the
thinking of low for the same answer. Pay for thinking only where a step needs it (lesson 14.3 picks effort per step).

## 5. Library version + check: your count next to the API's
`tiktoken` runs on your machine: it counts your own text for free, before you send it. `usage` comes back with the
reply and counts everything you are billed for. Your text matches; the wrapper and the hidden reasoning only show up in
`usage`.

In [12]:
print("tiktoken", tiktoken.__version__)
ours = n_sys + n_user
print("tiktoken, our two texts:", ours, "== 34 from the video:", ours == 34)
print("usage prompt_tokens:", prompt_t, "| wrapper (usage - ours):", prompt_t - ours)
print("rule: estimate with tiktoken before a call, trust usage after it")

tiktoken 0.14.0
tiktoken, our two texts: 34 == 34 from the video: True
usage prompt_tokens: 108 | wrapper (usage - ours): 74
rule: estimate with tiktoken before a call, trust usage after it


## 6. Where it shows up in agents
An agent is a loop of calls, and each step re-sends the whole, growing message list. So prompt tokens grow at every step
and you pay for early messages again and again. The cell replays one recorded Nightwatch incident from the course cache
(**no API call**) and prints the prompt tokens of each step. A `None` is a step whose tool call came back malformed, so no usage was billed.

In [13]:
import json
try:
    import agent_v0
    from brightcart.incidents import generate

    steps = []
    real_chat = agent_v0.chat
    def spy(body, mode="replay", sample=0):
        out = real_chat(body, "replay", sample)
        steps.append(out.get("usage", {}).get("prompt_tokens"))
        return out
    agent_v0.chat = spy
    run = agent_v0.run(next(x for x in generate() if x.id == "bad_deploy-2"), mode="replay")
    agent_v0.chat = real_chat
    print("prompt tokens per step:", steps)
    print("run usage:", run["usage"], f"| cost ${call_cost(run['usage']['prompt'], run['usage']['completion']):.6f}")
except (ImportError, LookupError) as e:
    print("replay not available here:", type(e).__name__, e)

prompt tokens per step: [693, 773, 1001, 1060, 1095, None, 1155, 1190, 1231]
run usage: {'prompt': 8198, 'completion': 941} | cost $0.000897


In [14]:
# All 180 recorded baseline runs of Nightwatch v0 (no API): how big is a typical run?
path = NIGHTWATCH / "runs" / "baseline_gpt-oss-20b.jsonl" if NIGHTWATCH else None
if path and path.exists():
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    tot = [r["usage"]["prompt"] + r["usage"]["completion"] for r in rows]
    share = sum(r["usage"]["prompt"] for r in rows) / sum(tot)
    print(f"{len(rows)} runs | mean {sum(tot) / len(tot):,.0f} tokens per run | prompt share {share:.0%}")
print(f"the video's round number, 11,000 tokens at 90% input: ${call_cost(9900, 1100):.6f}  (about a tenth of a cent)")

180 runs | mean 11,060 tokens per run | prompt share 87%
the video's round number, 11,000 tokens at 90% input: $0.001073  (about a tenth of a cent)


## 7. Traps
**Trap 1: counting only your own text.** The API also bills the wrapping.

In [15]:
try:
    assert prompt_t == n_sys + n_user, f"usage says {prompt_t}, our texts are {n_sys + n_user}: {prompt_t - n_sys - n_user} wrapper tokens"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: usage says 108, our texts are 34: 74 wrapper tokens


**Trap 2: trusting temperature 0 to give identical answers.** One passing test run proves very little.

In [16]:
try:
    assert len(set(hook_runs)) == 1, f"temperature 0 gave {len(set(hook_runs))} different answers in 8 runs"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: temperature 0 gave 2 different answers in 8 runs


**Trap 3: high effort everywhere.** Same answer `6`, far more thinking tokens (video numbers).

In [17]:
low, high = 39, 334
try:
    assert high <= 2 * low, f"high used {high / low:.1f}x the reasoning tokens of low for the same answer"
except AssertionError as e:
    print("AssertionError:", e)
print(f"extra cost of high over low, per call: ${(high - low) * PRICE_OUT / 1e6:.8f}, per million calls: ${(high - low) * PRICE_OUT:.2f}")

AssertionError: high used 8.6x the reasoning tokens of low for the same answer
extra cost of high over low, per call: $0.00008850, per million calls: $88.50


**Trap 4: pricing a call from the completion alone.** Prompt tokens are billed too, and in an agent they grow every step.

In [18]:
only_out = 102 * PRICE_OUT / 1e6
try:
    assert only_out == call_cost(108, 102), f"completion only ${only_out:.8f} vs real ${call_cost(108, 102):.8f}"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: completion only $0.00003060 vs real $0.00003870


## 8. Exercises

In [19]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

### Exercise 1 (from the video): your own alert, predicted and priced
Write `n_tokens(text)` that counts tokens with `enc`, and `price(usage)` that prices a usage dict with keys
`prompt_tokens` and `completion_tokens` in dollars. Then write your own alert in `MY_ALERT`.

In [20]:
MY_ALERT = "..."          # YOUR alert, e.g. a disk or a queue problem at Brightcart

def n_tokens(text):
    # YOUR CODE HERE
    return 0

def price(usage):
    # YOUR CODE HERE
    return 0.0

In [21]:
def test_1():
    assert n_tokens("checkout") == 1 and n_tokens("Brightcart") == 2, "use enc.encode and count the ids"
    assert n_tokens(ALERT) == 14, f"the lesson's alert is 14 tokens, got {n_tokens(ALERT)}"
    assert abs(price({"prompt_tokens": 108, "completion_tokens": 102}) - 0.0000387) < 1e-12, "108 in + 102 out = $0.0000387"
    assert abs(price({"prompt_tokens": 1_000_000, "completion_tokens": 0}) - 0.075) < 1e-12, "1M input tokens = $0.075"
    assert MY_ALERT != "..." and n_tokens(MY_ALERT) > 0, "write your own alert in MY_ALERT"

check(1, test_1)

✗ Exercise 1 failed: use enc.encode and count the ids


In [22]:
# calls the API (1 request): optional, send your alert and compare your guess with usage
if HAVE_KEY and MY_ALERT != "...":
    mine = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": MY_ALERT}]
    guess = n_tokens(SYSTEM) + n_tokens(MY_ALERT)
    r = client.chat.completions.create(model=MODEL, messages=mine, temperature=0)
    used = {"prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}
    print("tiktoken guess:", guess, "| usage:", used, f"| price ${price(used):.8f}")
else:
    print("skipped (no key, or MY_ALERT not written yet)")

skipped (no key, or MY_ALERT not written yet)


### Exercise 2: the wrapper
Write `wrapper_tokens(prompt_tokens, texts)`: the prompt tokens the API counted minus the tokens of your own texts.

In [23]:
def wrapper_tokens(prompt_tokens, texts):
    # YOUR CODE HERE
    return None

In [24]:
def test_2():
    assert wrapper_tokens(108, [SYSTEM, ALERT]) == 74, f"expected 74, got {wrapper_tokens(108, [SYSTEM, ALERT])}"
    assert wrapper_tokens(50, []) == 50, "with no texts, everything is wrapper"

check(2, test_2)

✗ Exercise 2 failed: expected 74, got None


### Exercise 3: an agent re-sends the list
Step 1 sends `first` prompt tokens. Every later step re-sends everything plus `added` new tokens. Write
`agent_prompt_tokens(first, added, steps)` that returns the **total** prompt tokens billed over all steps.

In [25]:
def agent_prompt_tokens(first, added, steps):
    # YOUR CODE HERE
    return 0

In [26]:
def test_3():
    assert agent_prompt_tokens(108, 50, 1) == 108, "one step: just the first prompt"
    assert agent_prompt_tokens(108, 50, 3) == 108 + 158 + 208, f"got {agent_prompt_tokens(108, 50, 3)}"
    assert agent_prompt_tokens(700, 60, 10) == 9700, f"got {agent_prompt_tokens(700, 60, 10)}"

check(3, test_3)

✗ Exercise 3 failed: one step: just the first prompt


### Exercise 4: is one test run enough?
Write `distinct_answers(replies)` that returns how many different replies a list holds, ignoring spaces at both ends.
Then `stable(replies)` returns True only when all replies are the same.

In [27]:
def distinct_answers(replies):
    # YOUR CODE HERE
    return 0

def stable(replies):
    # YOUR CODE HERE
    return True

In [28]:
def test_4():
    t1 = ["Payment gateway timeout.", "Payment gateway outage.", "Payment gateway timeout.",
          "Database timeout from traffic surge.", "Payment gateway outage. "]
    assert distinct_answers(t1) == 3, f"the video's temperature-1 run had 3 answers, got {distinct_answers(t1)}"
    assert distinct_answers(["Payment gateway outage."] * 5) == 1
    assert not stable(t1) and stable(["6", "6 ", " 6"]), "stable means one distinct answer"

check(4, test_4)

✗ Exercise 4 failed: the video's temperature-1 run had 3 answers, got 0


## Solutions (try first!)
### Solution 1

In [29]:
MY_ALERT = "Alert: disk on search-db is 95% full. What should I check first?"

def n_tokens(text):
    return len(enc.encode(text))

def price(usage):
    return usage["prompt_tokens"] / 1e6 * PRICE_IN + usage["completion_tokens"] / 1e6 * PRICE_OUT

### Solution 2

In [30]:
def wrapper_tokens(prompt_tokens, texts):
    return prompt_tokens - sum(len(enc.encode(t)) for t in texts)

### Solution 3

In [31]:
def agent_prompt_tokens(first, added, steps):
    total, size = 0, first
    for _ in range(steps):
        total += size          # this step re-sends everything so far
        size += added          # and the list grows before the next step
    return total

### Solution 4

In [32]:
def distinct_answers(replies):
    return len({r.strip() for r in replies})

def stable(replies):
    return distinct_answers(replies) == 1